In [4]:
import pandas as pd

MASTER = "/vol/biomedic3/awk24/datasets/Messidor2/messidor2_master.csv"
df = pd.read_csv(MASTER)


def grade_table(data: pd.DataFrame, index_col: str | None = None) -> pd.DataFrame:
    if index_col is None:
        counts = data["diagnosis"].value_counts().sort_index().rename("count").to_frame()
        counts["%"] = (counts["count"] / len(data) * 100).round(1)
        counts.index = [f"Grade {int(g)}" for g in counts.index]
        counts.index.name = "diagnosis"
        return counts
    pivot = (
        data.groupby([index_col, "diagnosis"])
        .size()
        .unstack(fill_value=0)
        .rename(columns=lambda g: f"Grade {int(g)}")
    )
    pivot["Total"] = pivot.sum(axis=1)

    return pivot


# ── Overall ───────────────────────────────────────────────────────────────────
print(f"Total images: {len(df)}\n")
print("=== Overall diagnosis split ===")
display(grade_table(df))

# ── By hospital ───────────────────────────────────────────────────────────────
print("\n=== Diagnosis split by hospital ===")
display(grade_table(df, "hospital"))

# ── By eye drops ─────────────────────────────────────────────────────────────
print("\n=== Diagnosis split by with_eye_drops ===")
df_disp = df.copy()
df_disp["with_eye_drops"] = df_disp["with_eye_drops"].map({0: "no", 1: "yes"}).fillna("unknown")
display(grade_table(df_disp, "with_eye_drops"))


print("\n=== Diagnosis split by format ===")
display(grade_table(df, "format"))

Total images: 1744

=== Overall diagnosis split ===


,count,%
diagnosis,,
Grade 0,1017,58.3
Grade 1,270,15.5
Grade 2,347,19.9
Grade 3,75,4.3
Grade 4,35,2.0



=== Diagnosis split by hospital ===


diagnosis,Grade 0,Grade 1,Grade 2,Grade 3,Grade 4,Total
hospital,,,,,,
Brest,716,113,122,8,16,975
Etienne,160,109,107,21,0,397
Lariboisiere,141,48,118,46,19,372



=== Diagnosis split by with_eye_drops ===


diagnosis,Grade 0,Grade 1,Grade 2,Grade 3,Grade 4,Total
with_eye_drops,,,,,,
no,716,113,122,8,16,975
yes,301,157,225,67,19,769



=== Diagnosis split by format ===


diagnosis,Grade 0,Grade 1,Grade 2,Grade 3,Grade 4,Total
format,,,,,,
jpg,549,63,57,4,14,687
png,468,207,290,71,21,1057


In [2]:
import json
from pathlib import Path
import pandas as pd


SPLITS_DIR  = Path("/vol/biomedic3/awk24/datasets/Messidor2")
EXPERIMENTS = ["dilated", "nondilated", "all"]

df_master = pd.read_csv("/vol/biomedic3/awk24/datasets/Messidor2/messidor2_master.csv")
diag_map  = dict(zip(df_master["id_code"], df_master["diagnosis"]))
pair_map  = dict(zip(df_master["id_code"], df_master["pair_id"]))


def grade_counts_table(splits: dict[str, list[str]]) -> pd.DataFrame:
    """Count and % per grade for each split side by side."""
    rows = []
    for g in range(5):
        row = {"grade": f"Grade {g}"}
        for split_name, imgs in splits.items():
            cnt = sum(1 for img in imgs if diag_map.get(img) == g)
            pct = 100 * cnt / len(imgs) if imgs else 0.0
            row[f"{split_name} n"] = cnt
            row[f"{split_name} %"] = round(pct, 1)
        rows.append(row)
    return pd.DataFrame(rows).set_index("grade")


def check_no_patient_overlap(splits: dict[str, list[str]]) -> list[str]:
    """Return error strings for any patient (pair_id) that appears in more than one split."""
    pair_to_splits: dict = {}
    for split_name, imgs in splits.items():
        for img in imgs:
            pid = pair_map.get(img)
            if pid is None:
                continue
            pair_to_splits.setdefault(pid, set()).add(split_name)
    return [
        f"patient pair_id={pid} appears in {sorted(split_set)}"
        for pid, split_set in pair_to_splits.items()
        if len(split_set) > 1
    ]


for exp in EXPERIMENTS:
    with open(SPLITS_DIR / f"messidor2_splits_{exp}.json") as f:
        data = json.load(f)

    ref       = data["reference_pool"]
    test      = data["test"]
    val       = data["validation"]
    orderings = data["hospital_b_orderings"]

    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {exp.upper()}")
    print(f"  reference_pool: {len(ref)}  |  test: {len(test)}  |  validation: {len(val)}")
    print(f"{'='*60}")

    # ── Diagnosis split per subset ────────────────────────────────────────────
    display(grade_counts_table({"reference_pool": ref, "test": test, "validation": val}))

    # ── Patient overlap check ─────────────────────────────────────────────────
    overlap_errors = check_no_patient_overlap({"reference_pool": ref, "test": test, "validation": val})
    if overlap_errors:
        print(f"  Patient overlap ERRORS ({len(overlap_errors)}):")
        for e in overlap_errors:
            print(f"    {e}")
    else:
        print("  Patient overlap check: OK (no patient appears in more than one split)")

    # ── hospital_b ordering seed checks ──────────────────────────────────────
    print("  hospital_b_orderings — correctness checks:")
    ref_set = set(ref)
    all_ok  = True

    for seed_name, grade_lists in orderings.items():
        errors = []

        for grade_key, imgs in grade_lists.items():
            expected = int(grade_key[1])  # "G0" -> 0
            for img in imgs:
                if img not in ref_set:
                    errors.append(f"{img} not in reference_pool")
                elif diag_map.get(img) != expected:
                    errors.append(f"{img}: labelled {grade_key} but diagnosis={diag_map.get(img)}")

        all_in_seed = [img for imgs in grade_lists.values() for img in imgs]
        if len(all_in_seed) != len(set(all_in_seed)):
            errors.append("duplicate images within ordering")
        if sorted(all_in_seed) != sorted(ref):
            errors.append(f"coverage mismatch: {len(all_in_seed)} in ordering vs {len(ref)} in reference_pool")

        seed_ok    = len(errors) == 0
        all_ok     = all_ok and seed_ok
        counts_str = "  ".join(f"{k}={len(v)}" for k, v in sorted(grade_lists.items()))
        status     = "OK" if seed_ok else f"FAIL — {errors[:3]}"
        print(f"    {seed_name}: {status}  [{counts_str}]")

    if all_ok:
        print("    All seeds passed.")



EXPERIMENT: DILATED
  reference_pool: 389  |  test: 306  |  validation: 74


,reference_pool n,reference_pool %,test n,test %,validation n,validation %
grade,,,,,,
Grade 0,155,39.8,116,37.9,30,40.5
Grade 1,78,20.1,61,19.9,18,24.3
Grade 2,114,29.3,93,30.4,18,24.3
Grade 3,33,8.5,28,9.2,6,8.1
Grade 4,9,2.3,8,2.6,2,2.7


  Patient overlap check: OK (no patient appears in more than one split)
  hospital_b_orderings — correctness checks:
    seed_A: OK  [G0=155  G1=78  G2=114  G3=33  G4=9]
    seed_B: OK  [G0=155  G1=78  G2=114  G3=33  G4=9]
    seed_C: OK  [G0=155  G1=78  G2=114  G3=33  G4=9]
    All seeds passed.

EXPERIMENT: NONDILATED
  reference_pool: 487  |  test: 390  |  validation: 98


,reference_pool n,reference_pool %,test n,test %,validation n,validation %
grade,,,,,,
Grade 0,360,73.9,285,73.1,71,72.4
Grade 1,59,12.1,42,10.8,12,12.2
Grade 2,57,11.7,53,13.6,12,12.2
Grade 3,4,0.8,3,0.8,1,1.0
Grade 4,7,1.4,7,1.8,2,2.0


  Patient overlap check: OK (no patient appears in more than one split)
  hospital_b_orderings — correctness checks:
    seed_A: OK  [G0=360  G1=59  G2=57  G3=4  G4=7]
    seed_B: OK  [G0=360  G1=59  G2=57  G3=4  G4=7]
    seed_C: OK  [G0=360  G1=59  G2=57  G3=4  G4=7]
    All seeds passed.

EXPERIMENT: ALL
  reference_pool: 871  |  test: 698  |  validation: 175


,reference_pool n,reference_pool %,test n,test %,validation n,validation %
grade,,,,,,
Grade 0,510,58.6,408,58.5,99,56.6
Grade 1,135,15.5,108,15.5,27,15.4
Grade 2,170,19.5,140,20.1,37,21.1
Grade 3,37,4.2,30,4.3,8,4.6
Grade 4,19,2.2,12,1.7,4,2.3


  Patient overlap check: OK (no patient appears in more than one split)
  hospital_b_orderings — correctness checks:
    seed_A: OK  [G0=510  G1=135  G2=170  G3=37  G4=19]
    seed_B: OK  [G0=510  G1=135  G2=170  G3=37  G4=19]
    seed_C: OK  [G0=510  G1=135  G2=170  G3=37  G4=19]
    All seeds passed.


In [7]:
import json
with open("/vol/biomedic3/awk24/datasets/Messidor2/messidor2_splits_dilated.json") as f:
    d = json.load(f)
print(set(d["validation"]) & set(d["test"]))     # should be empty
print(set(d["validation"]) & set(d["reference_pool"]))  # should be empty
print(set(d["test"]) & set(d["reference_pool"]))  # should be empty

set()
set()
set()
